In [1]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import math
import matplotlib.animation

plt.rcParams["animation.html"] = "jshtml"
plt.rcParams['figure.dpi'] = 150

# Definizione dei parametri

In [2]:
# Variabili simboliche dei giunti (giunti e velocità)
sp.var(
    'q1 q2 q3 q4 dot_q1 dot_q2 dot_q3 dot_q4',
    real=True
)
# Parametri geometrici dei link
d1 = 0.03
d2 = 0.105
# Lunghezze (metri)
l1, l2, l3, l4 = 0.065, 0.065, np.sqrt(d1**2+d2**2), 0.030
a = np.array([l1, l2, 0.0, l4])
d = np.array([0, 0, -d1, d2])
alpha = np.array([-np.pi/2, np.pi/2, -np.pi/2, 0])

# Dimensioni della sezione trasversale rettangolare dei link (altezza a, larghezza b) in metri
c1, b1 = 0.040, 0.030
c2, b2 = 0.040, 0.007
c3, b3 = 0.004, 0.004
c4, b4 = 0.034, 0.025

# Dimensioni geometriche del rotore (altezza h, raggio r) in metri
h_grande, r_grande = 0.024, 0.012
h_medio, r_medio = 0.024, 0.002

# Masse (kg)
ml1, ml2, ml3, ml4 = 0.020, 0.012, 0.046, 0.002   # masse dei link
mr1, mr2, mr3, mr4 = 0.060, 0.060, 0.035, 0.035   # masse degli statori
Mm1, Mm2, Mm3, Mm4 = 0.015, 0.015, 0.005, 0.005   # masse dei rotori

# Rapporti di trasmissione dei motori
kr1, kr2, kr3, kr4 = 1, 1, 1, 1

# Masse totali dei link
# Link 1 supporta: sé stesso + suo motore + statore del motore 2
M1 = ml1 + Mm1 + mr2
# Link 2 supporta: sé stesso + suo motore + statore del motore 3
M2 = ml2 + Mm2 + mr3
# Link 3 supporta: sé stesso + suo motore + statore del motore 4
M3 = ml3 + Mm3 + mr4
# Link 4 supporta: sé stesso + suo motore
M4 = ml4 + Mm4


### Calcolo del centro di massa

In [3]:
# Centro di massa dei link
COM_1 = sp.Matrix([-l1/2, 0, 0, 1])  # rispetto alla Terna1
COM_2 = sp.Matrix([-l2/2, 0, 0, 1])  # rispetto alla Terna2
COM_3 = sp.Matrix([0, -d1/2, d2/2, 1])   # rispetto alla Terna3
COM_4 = sp.Matrix([-l4/2, 0, 0, 1])  # rispetto alla Terna4

# Centro di massa dei motori
COM_m1 = sp.Matrix([0, 0, 0])  # rispetto alla Terna0
COM_m2 = sp.Matrix([0, 0, 0])  # rispetto alla Terna1
COM_m3 = sp.Matrix([0, 0, 0])  # rispetto alla Terna2
COM_m4 = sp.Matrix([0, 0, 0.105])  # rispetto alla Terna3

### Calcolo dei tensori di Inerzia

In [4]:
# Tensori di Inerzia dei Link [kg·m²]
# Inerzia dei link rispetto al proprio centro di massa
# Assunzione: il link è un corpo rigido rettangolare
I_1 = sp.Matrix([
    [1/12 * M1 * (c1**2 + b1**2), 0, 0],    # Ixx
    [0, 1/12 * M1 * (b1**2 + l1**2), 0],    # Iyy
    [0, 0, 1/12 * M1 * (c1**2 + l1**2)]     # Izz
])
I_2 = sp.Matrix([
    [1/12 * M2 * (c2**2 + b2**2), 0, 0],
    [0, 1/12 * M2 * (b2**2 + l2**2), 0],
    [0, 0, 1/12 * M2 * (c2**2 + l2**2)]
])
I_3 = sp.Matrix([
    [1/12 * M3 * (c3**2 + b3**2), 0, 0],
    [0, 1/12 * M3 * (b3**2 + l3**2), 0],
    [0, 0, 1/12 * M3 * (c3**2 + l3**2)]
])
I_4 = sp.Matrix([
    [1/12 * M4 * (c4**2 + b4**2), 0, 0],
    [0, 1/12 * M4 * (b4**2 + l4**2), 0],
    [0, 0, 1/12 * M4 * (c4**2 + l4**2)]
])
# Tensori di Inerzia dei Motori [kg·m²]
# I motori sono modellati come cilindri solidi:
# Inerzia dei motori nel proprio centro di massa e sistema di riferimento
I_m1_b = sp.Matrix([
    [1/12 * Mm1 * (h_grande**2 + 3*r_grande**2), 0, 0],
    [0, 1/12 * Mm1 * (h_grande**2 + 3*r_grande**2), 0],
    [0, 0, 1/2 * Mm1 * r_grande**2]
])
I_m2_b = sp.Matrix([
    [1/12 * Mm2 * (h_grande**2 + 3*r_grande**2), 0, 0],
    [0, 1/12 * Mm2 * (h_grande**2 + 3*r_grande**2), 0],
    [0, 0, 1/2 * Mm2 * r_grande**2]
])
I_m3_b = sp.Matrix([
    [1/12 * Mm3 * (h_medio**2 + 3*r_medio**2), 0, 0],
    [0, 1/12 * Mm3 * (h_medio**2 + 3*r_medio**2), 0],
    [0, 0, 1/2 * Mm3 * r_medio**2]
])
I_m4_b = sp.Matrix([
    [1/12 * Mm4 * (h_medio**2 + 3*r_medio**2), 0, 0],
    [0, 1/12 * Mm4 * (h_medio**2 + 3*r_medio**2), 0],
    [0, 0, 1/2 * Mm4 * r_medio**2]
])

# Risoluzione simbolica della Cinematica

In [5]:
# Risoluzione simbolica della Cinematica

# Vettore delle coordinate articolari (posizioni angolari)
Q = sp.Matrix([q1, q2, q3, q4])

# Vettore delle velocità articolari (derivate rispetto al tempo)
Q_dot = sp.Matrix([dot_q1, dot_q2, dot_q3, dot_q4])

# -------------------------
# Definizione della funzione per calcolare la matrice di trasformazione omogenea
# secondo i parametri Denavit-Hartenberg simbolici
# -------------------------
def DH_sym(a, d, alpha, theta):
    T = sp.Matrix([
        [sp.cos(theta), -sp.sin(theta) * sp.cos(alpha),  sp.sin(theta) * sp.sin(alpha), a * sp.cos(theta)],
        [sp.sin(theta),  sp.cos(theta) * sp.cos(alpha), -sp.cos(theta) * sp.sin(alpha), a * sp.sin(theta)],
        [0,              sp.sin(alpha),                 sp.cos(alpha),                 d],
        [0,              0,                             0,                             1]
    ])
    return T

# Calcolo delle matrici di trasformazione omogenea
T_b0 = DH_sym(0, 0, np.pi/2, 0)
T_01 = DH_sym(a[0], d[0], alpha[0], q1)
T_12 = DH_sym(a[1], d[1], alpha[1], q2)
T_23 = DH_sym(a[2], d[2], alpha[2], q3)
T_34 = DH_sym(a[3], d[3], alpha[3], q4)

T_b1 = T_b0 * T_01
T_b2 = T_b1 * T_12
T_b3 = T_b2 * T_23
T_b4 = T_b3 * T_34

# Estrazione degli assi z (orientamento dei giunti)
z_0 = sp.Matrix([0, 0, 1])
z_1 = T_b1[:3, 2]
z_2 = T_b2[:3, 2]
z_3 = T_b3[:3, 2]
z_4 = T_b4[:3, 2]

# Posizione dei frame rispetto alla base
p_0 = sp.Matrix([0, 0, 0])
p_1 = T_b1[:3, 3]
p_2 = T_b2[:3, 3]
p_3 = T_b3[:3, 3]
p_4 = T_b4[:3, 3]



In [ ]:
T_b4

### Definizione del vettore gravitazionale in terna base

In [ ]:
# Vettore Gravitazionale [m/s²]

# Definizione del vettore di accelerazione gravitazionale nel sistema di riferimento base
g0 = sp.Matrix([0, -9.81, 0])
R_b0 = T_b0[:3, :3]  # matrice di rotazione da 0 a base
g_b = R_b0 * g0   # gravità espressa nella base

# Calcolo degli Jacobiani

### Link 1

In [26]:
# Posizione del centro di massa del Link 1 in terna base

P1_ = T_b1 * sp.Matrix(COM_1)
P1 = sp.Matrix(P1_[:3])  # centro di massa del link 1 in base

# Tensore d'inerzia del Link 1 riportato in terna base

I_b1 = T_b1[:3, :3] * I_1 * T_b1[:3, :3].T

# Jacobiana lineare del Link 1 rispetto al centro di massa

JP1_l1 = sp.Matrix.cross(z_0, P1 - p_0)
JP_l1 = sp.Matrix.hstack(JP1_l1, sp.zeros(3, 3))

# Jacobiana angolare del Link 1

JO1_l1 = z_0
JO_l1 = sp.Matrix.hstack(JO1_l1, sp.zeros(3, 3))


### Link 2

In [27]:
# Posizione del centro di massa del Link 2 in terna base

P2_ = T_b2 * sp.Matrix(COM_2)
P2 = sp.Matrix(P2_[:3])  # centro di massa del link 2 in base

# Tensore di inerzia del Link 2 riportato in terna base

I_b2 = T_b2[:3, :3] * I_2 * T_b2[:3, :3].T

# Jacobiana lineare del Link 2 rispetto al centro di massa

JP1_l2 = sp.Matrix.cross(z_0, P2)
JP2_l2 = sp.Matrix.cross(z_1, P2 - p_1)
JP_l2 = sp.Matrix.hstack(JP1_l2, JP2_l2, sp.zeros(3, 2))

# Jacobiana angolare del Link 2

JO1_l2 = z_0
JO2_l2 = z_1
JO_l2 = sp.Matrix.hstack(JO1_l2, JO2_l2, sp.zeros(3, 2))

### Link 3

In [28]:
# Posizione del centro di massa del Link 3 in terna base

P3_ = T_b3 * sp.Matrix(COM_3)
P3 = sp.Matrix(P3_[:3])  # centro di massa del link 3 in base

# Tensore di inerzia del Link 3 riportato in terna base

I_b3 = T_b3[:3, :3] * I_3 * T_b3[:3, :3].T

# Jacobiana lineare del Link 3 rispetto al centro di massa

JP1_l3 = sp.Matrix.cross(z_0, P3)
JP2_l3 = sp.Matrix.cross(z_1, P3 - p_1)
JP3_l3 = sp.Matrix.cross(z_2, P3 - p_2)
JP_l3 = sp.Matrix.hstack(JP1_l3, JP2_l3, JP3_l3, sp.zeros(3, 1))

# Jacobiana angolare del Link 3

JO1_l3 = z_0
JO2_l3 = z_1
JO3_l3 = z_2
JO_l3 = sp.Matrix.hstack(JO1_l3, JO2_l3, JO3_l3, sp.zeros(3, 1))

### Link 4

In [29]:
# Posizione del centro di massa del Link 4 in terna base

P4_ = T_b4 * sp.Matrix(COM_4)
P4 = sp.Matrix(P4_[:3])  # centro di massa del link 4 in base

# Tensore di inerzia del Link 4 riportato in terna base

I_b4 = T_b4[:3, :3] * I_4 * T_b4[:3, :3].T

# Jacobiana lineare del Link 4 rispetto al centro di massa

JP1_l4 = sp.Matrix.cross(z_0, P4)
JP2_l4 = sp.Matrix.cross(z_1, P4 - p_1)
JP3_l4 = sp.Matrix.cross(z_2, P4 - p_2)
JP4_l4 = sp.Matrix.cross(z_3, P4 - p_3)
JP_l4 = sp.Matrix.hstack(JP1_l4, JP2_l4, JP3_l4, JP4_l4)

# Jacobiana angolare del Link 4

JO1_l4 = z_0
JO2_l4 = z_1
JO3_l4 = z_2
JO4_l4 = z_3
JO_l4 = sp.Matrix.hstack(JO1_l4, JO2_l4, JO3_l4, JO4_l4)

### Motori

In [30]:
# MOTORE 1 (fissato alla base)

# Centro di massa del motore 1 nel sistema base
pm1 = T_b0[:3, :3] * COM_m1

# Tensore di inerzia del motore 1
I_m1= T_b0[:3, :3] * I_m1_b * T_b0[:3, :3].T

# Jacobiana lineare del motore 1: non contribuisce al moto traslazionale del braccio (fissato alla base)
# quindi è una matrice nulla (nessuna influenza sul centro di massa del manipolatore)
JP_m1 = sp.zeros(3, 4)

# Jacobiana angolare del motore 1:
# - Solo il primo giunto contribuisce
JO1_m1 = kr1 * z_0
JO_m1 = sp.Matrix.hstack(JO1_m1, sp.zeros(3, 3))

# MOTORE 2 (montato sul link 1, espresso nella Terna 1)

# Centro di massa del motore 2 espresso in base:
pm2 = T_b1[:3, :3] * COM_m2

# Tensore di inerzia del motore 2 riportato in terna base:
I_m2 = T_b1[:3, :3] * I_m2_b * T_b1[:3, :3].T

# Jacobiana lineare del motore 2:
# - Il centro di massa è influenzato solo dal primo giunto (motore è fissato sul link 1)
# - Si usa la classica formula z_i × (p_com - p_i)
JP1_m2 = sp.Matrix.cross(z_0, pm2 - p_0)
JP_m2 = sp.Matrix.hstack(JP1_m2, sp.zeros(3, 3))  # solo il primo giunto agisce sulla posizione

# Jacobiana angolare del motore 2:
JO2_m2 = kr2 * z_1
JO_m2 = sp.Matrix.hstack(JO1_l2, JO2_m2, sp.zeros(3, 2))

# MOTORE 3 (montato sul link 2, espresso nella Terna 2)

# Centro di massa del motore 3 espresso in base:
pm3 = T_b2[:3, :3] * COM_m3

# Tensore di inerzia del motore 3 in base:
I_m3 = T_b2[:3, :3] * I_m3_b * T_b2[:3, :3].T

# Jacobiana lineare:
# - Influenzato dai giunti 1 e 2
JP1_m3 = sp.Matrix.cross(z_0, pm3 - p_0)
JP2_m3 = sp.Matrix.cross(z_1, pm3 - p_1)
JP_m3 = sp.Matrix.hstack(JP1_m3, JP2_m3, sp.zeros(3, 2))  # solo primi 2 giunti agiscono

# Jacobiana angolare:
JO3_m3 = kr3 * z_2
JO_m3 = sp.Matrix.hstack(JO1_l3, JO2_l3, JO3_m3, sp.zeros(3, 1))

# MOTORE 4 (montato sul link 3, espresso nella Terna 3)

# Centro di massa del motore 4 espresso in base:
pm4 = T_b3[:3, :3] * COM_m4

# Tensore di inerzia del motore 4 in base:
I_m4 = T_b3[:3, :3] * I_m4_b * T_b3[:3, :3].T

# Jacobiana lineare:
# - Influenzato dai giunti 1, 2, 3
JP1_m4 = sp.Matrix.cross(z_0, pm4 - p_0)
JP2_m4 = sp.Matrix.cross(z_1, pm4 - p_1)
JP3_m4 = sp.Matrix.cross(z_2, pm4 - p_2)
JP_m4 = sp.Matrix.hstack(JP1_m4, JP2_m4, JP3_m4, sp.zeros(3, 1))

# Jacobiana angolare:
JO4_m4 = kr4 * z_3
JO_m4 = sp.Matrix.hstack(JO1_l4, JO2_l4, JO3_l4, JO4_m4)



# Calcolo termini dinamici

### Matrice di Inerzia B

In [31]:
# Matrice di Inerzia del Sistema: B(q)

# LINK 1

# Contributo alla matrice di inerzia dovuto al link 1:
# - Parte traslazionale: M_1 * Jp.T * Jp
# - Parte rotazionale:   JO.T * I * JO
B1_l = M1 * JP_l1.T * JP_l1 + JO_l1.T * I_b1 * JO_l1
B1_m = Mm1 * JP_m1.T * JP_m1 + JO_m1.T * I_m1 * JO_m1
B1 = B1_l + B1_m

# LINK 2
B2_l = M2 * JP_l2.T * JP_l2 + JO_l2.T * I_b2 * JO_l2
B2_m = Mm2 * JP_m2.T * JP_m2 + JO_m2.T * I_m2 * JO_m2
B2 = B2_l + B2_m

# LINK 3
B3_l = M3 * JP_l3.T * JP_l3 + JO_l3.T * I_b3 * JO_l3
B3_m = Mm3 * JP_m3.T * JP_m3 + JO_m3.T * I_m3 * JO_m3
B3 = B3_l + B3_m

# LINK 4
B4_l = M4 * JP_l4.T * JP_l4 + JO_l4.T * I_b4 * JO_l4
B4_m = Mm4 * JP_m4.T * JP_m4 + JO_m4.T * I_m4 * JO_m4
B4 = B4_l + B4_m

# MATRICE DI INERZIA COMPLESSIVA

# Somma di tutti i contributi: link e motori da 1 a 4
B = B1 + B2 + B3 + B4  # Matrice di inerzia del sistema B(q)

### Matrice C

In [33]:
# -------------------------
# Matrice di Coriolis: C(q, q̇)
# -------------------------

# Inizializzazione della matrice C (4x4) con zeri simbolici
C = sp.zeros(4)

# Loop triplo per calcolare ogni elemento C[i, j]
for i in range(len(Q)):        # riga i della matrice C
    for j in range(len(Q)):    # colonna j della matrice C
        for k in range(len(Q)):  # somma su k (indice del termine moltiplicato per q̇_k)
            # Derivate parziali dei termini della matrice di inerzia B
            a = sp.diff(B[i, j], Q[k])  # ∂B_ij / ∂q_k
            b = sp.diff(B[i, k], Q[j])  # ∂B_ik / ∂q_j
            c = sp.diff(B[j, k], Q[i])  # ∂B_jk / ∂q_i

            # Formula dei coefficienti di Coriolis (Christoffel symbols)
            # C_ij += 0.5 * (a + b - c) * q̇_k
            C[i, j] += 0.5 * (a + b - c) * Q_dot[k]

### Coppie Gravitazionali G

In [35]:
# Inizializzazione dei vettori
G = sp.zeros(4, 1)      # Vettore risultante di gravità
g_m = sp.zeros(4, 1)    # Contributi gravitazionali dei motori
g_l = sp.zeros(4, 1)    # Contributi gravitazionali dei link

# Loop sui giunti per calcolare ciascun termine G[i]
for i in range(len(Q)):
    # --- Contributo dei motori ---
    # Forza peso = massa * g
    # Torque gravitazionale = J^T * (massa * g)
    # JP_m1 e JP_m2 sono le Jacobiane lineari dei motori rispetto ai giunti
    g_m[i] = Mm1 * g_b.T * JP_m1[:, i] + Mm2 * g_b.T * JP_m2[:, i] + Mm3 * g_b.T * JP_m3[:, i] + Mm4 * g_b.T * JP_m4[:, i]

    # Contributo dei link
    g_l[i] = M1 * g_b.T * JP_l1[:, i] + M2 * g_b.T * JP_l2[:, i] + M3 * g_b.T * JP_l3[:, i] + M4 * g_b.T * JP_l4[:, i]

    # Somma dei contributi con segno negativo
    G[i] = -(g_l[i] + g_m[i])

### Coppie di attrito

In [36]:
# Modello di Attrito Viscoso (Dinamico)

# Vettore delle coordinate articolari (posizioni angolari)
Q = sp.Matrix([q1, q2, q3, q4])

# Vettore delle velocità articolari (derivate rispetto al tempo)
Q_dot = sp.Matrix([dot_q1, dot_q2, dot_q3, dot_q4])

# Matrice dei coefficienti di attrito viscoso (lineare rispetto a q̇)
# In questo caso, si assume attrito uguale su tutti i giunti: 0.05 N·m·s/rad
Fv = sp.Matrix([
    [0.5, 0.0, 0.0, 0.0],
    [0.0, 0.5, 0.0, 0.0],
    [0.0, 0.0, 0.5, 0.0],
    [0.0, 0.0, 0.0, 0.5]
]) * 1e-1

# Coppie viscose: tau_v = Fv * q̇
tau_v = Fv * Q_dot

# Modello di Attrito Statico (Coulomb)

# Matrice dei coefficienti di attrito statico
# Valore tipico: 0.01 N·m (coppia resistente costante che si oppone al moto)
Fs = sp.Matrix([
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 1.0]
]) * 1e-2

# Vettore del segno di q̇: restituisce +1, -1 o 0 per ogni giunto
signQ_dot = sp.Matrix([
    sp.sign(Q_dot[0]),
    sp.sign(Q_dot[1]),
    sp.sign(Q_dot[2]),
    sp.sign(Q_dot[3])
])

# Coppie statiche: tau_s = Fs * sign(q̇)
tau_s = Fs * signQ_dot

# Modello Dinamico Diretto

In [37]:
# Definizione simbolica delle variabili per torque e accelerazioni

# Coppie ai giunti (input al sistema)
tau1, tau2, tau3, tau4 = sp.symbols('tau1 tau2 tau3 tau4', real = True)
# Accelerazioni articolari (output del modello dinamico diretto)
q2dot1, q2dot2, q2dot3, q2dot4 = sp.symbols('q2dot1 q2dot2 q2dot3 q2dot4', real = True)

# Vettore delle coppie simboliche
tau = sp.Matrix([tau1, tau2, tau3, tau4])

# Vettore delle accelerazioni simboliche
Q2dot = sp.Matrix([q2dot1, q2dot2, q2dot3, q2dot4])

# Modello Dinamico Diretto

# Forma inversa dell'equazione del moto:
# B(q) * q̈ + C(q, q̇) * q̇ + G(q) + τ_v + τ_s = τ
# ⇒ q̈ = B⁻¹ * (τ - C * q̇ - τ_v - τ_s - G)
#Q2dot_expr = B.inv() * (tau - C * Q_dot - tau_v - tau_s - G)
Q2dot_expr = B.LUsolve(tau - C*Q_dot - tau_v - tau_s - G)

### Esporto la funzione per il calcolo del modello dinamico

In [38]:
# -------------------------
# Funzione numerica per il modello dinamico diretto
# -------------------------
import sys
# Alza il limite di ricorsione per evitare errori su espressioni molto profonde
sys.setrecursionlimit(10000)
# Q2dot_expr_np:
# Questa funzione restituisce le accelerazioni articolari q̈ = f(τ, q, q̇)
# Input: tau1, tau2, tau3, tau4, q1, q2, q3, q4, q̇1, q̇2, q̇3, q̇4
# Output: accelerazioni [q̈1, q̈2, q̈3, q̈4]
Q2dot_expr_np = sp.lambdify(
    [
        tau[0], tau[1], tau[2], tau[3],       # τ1…τ4
        Q[0],   Q[1],   Q[2],   Q[3],         # q1…q4
        Q_dot[0], Q_dot[1], Q_dot[2], Q_dot[3] # q̇1…q̇4
    ],
    Q2dot_expr.T,
    'numpy',
    cse=True,             # abilita CommonSubexpression Elimination
)


In [39]:
# Funzione per la matrice di inerzia B(q)
B_expr_np = sp.lambdify([Q[0], Q[1], Q[2], Q[3]], B, 'numpy')

# Funzione per la matrice di Coriolis C(q, q̇)
C_expr_np = sp.lambdify(
    [Q[0], Q[1], Q[2], Q[3], Q_dot[0], Q_dot[1], Q_dot[2], Q_dot[3]], C,'numpy')

# Funzione per il vettore gravitazionale G(q)
G_expr_np = sp.lambdify([Q[0], Q[1], Q[2], Q[3]], G.T, 'numpy')

# Funzioni per attriti
tau_v_expr_np = sp.lambdify(
    [Q_dot[0], Q_dot[1], Q_dot[2], Q_dot[3]], tau_v, 'numpy')

tau_s_expr_np = sp.lambdify(
    [Q_dot[0], Q_dot[1], Q_dot[2], Q_dot[3]], tau_s, 'numpy')

In [ ]:
# Esempio d'uso (numerico):
tau_val = [1.0, 0.5, -0.3, 0.2]
q_val = [0.5, 0.0, -0.2, 0.3]
q_dot_val = [0.0, 0.0, 0.1, -0.1]

# Calcolo accelerazioni articolari
q_ddot = Q2dot_expr_np(*tau_val, *q_val, *q_dot_val)[0]
print("Accelerazioni:", q_ddot)

# Calcolo della gravità
g_val = G_expr_np(*q_val)[0]
print("Gravità:", g_val)

# Simulazione dinamica

In [41]:
# Definizione funzione cinematica diretta secondo convenzione DH
def DH(d, a, alpha, theta):
    T = np.array([
        [np.cos(theta), -np.sin(theta)*np.cos(alpha), np.sin(theta)*np.sin(alpha), a*np.cos(theta)],
        [np.sin(theta), np.cos(theta)*np.cos(alpha), -np.cos(theta)*np.sin(alpha), a*np.sin(theta)],
        [0.0,           np.sin(alpha),               np.cos(alpha),               d],
        [0.0,           0.0,                         0.0,                         1.0]
    ])
    return T

In [ ]:
# Parametri geometrici dei link
d1 = 0.03
d2 = 0.105
# Lunghezze (metri)
l1, l2, l3, l4 = 0.065, 0.065, np.sqrt(d1**2+d2**2), 0.030
a = np.array([l1, l2, 0.0, l4])
d = np.array([0, 0, -d1, d2])
alpha = np.array([-np.pi/2, np.pi/2, -np.pi/2, 0])

# Stato iniziale: posizione, velocità, accelerazione
Q = np.array([0.0, 0.0, 0.0, 0.0])        # Posizioni articolari iniziali [rad]

Qd = np.array([0, np.pi/5, np.pi/4, np.pi])  # Posizioni articolari desiderate [rad]

Qdot = np.array([0.0, 0.0, 0.0, 0.0])     # Velocità articolari iniziali [rad/s]
Q2dot = np.array([0.0, 0.0, 0.0, 0.0])    # Accelerazioni articolari iniziali [rad/s^2]
tau = np.array([0.0, 0.0, 0.0, 0.0])      # Coppie applicate iniziali [Nm]

# Liste per memorizzare l'evoluzione temporale
Q2dot_ = []  # Accelerazioni
Qdot_ = []   # Velocità
Q_ = []      # Posizioni

# Tracciamento traiettoria nello spazio
XYZ1 = []     # Posizione del giunto 2
XYZ2 = []     # Posizione del giunto 3
XYZ3 = []     # Posizione del giunto 4
XYZee = []     # Posizione del end-effector

# Costante proporzionale per il controllo in posizione
Kp = 0.5
Kd = 0.2

# -------------------------
# Simulazione temporale
# -------------------------

dt = 0.001  # Passo di integrazione [s]
time = np.arange(0, 5, dt)  # Tempo totale: 5 secondi
Tb0 = DH(0, 0, np.pi/2, 0)

for t in time:

    # Controllo: PD + compensazione della gravità
    tau = Kp * (Qd - Q) - Kd * (Qdot) + G_expr_np(*Q)[0]

    # Dinamica diretta: calcolo accelerazioni articolari
    Q2dot = Q2dot_expr_np(*tau, *Q, *Qdot)[0]

    # Integrazione semplice (Eulero)
    Qdot = Qdot + Q2dot * dt
    Q = Q + Qdot * dt

    # Salvataggio dati
    Q2dot_.append([Q2dot[0], Q2dot[1], Q2dot[2], Q2dot[3]])
    Qdot_.append([Qdot[0], Qdot[1], Qdot[2], Qdot[3]])
    Q_.append([Q[0], Q[1], Q[2], Q[3]])

    # Calcolo trasformazioni tra i frame
    T01 = DH(d[0], a[0], alpha[0], Q[0])
    T12 = DH(d[1], a[1], alpha[1], Q[1])
    T23 = DH(d[2], a[2], alpha[2], Q[2])
    T34 = DH(d[3], a[3], alpha[3], Q[3])

    Tb1 = np.matmul(Tb0,T01)
    Tb2 = np.matmul(Tb1,T12)
    Tb3 = np.matmul(Tb2,T23)
    Tb4 = np.matmul(Tb3,T34)

    # Salva coordinate XY
    XYZ1.append([Tb1[0, 3], Tb1[1, 3], Tb1[2, 3]])
    XYZ2.append([Tb2[0, 3], Tb2[1, 3], Tb2[2, 3]])
    XYZ3.append([Tb3[0, 3], Tb3[1, 3], Tb3[2, 3]])
    XYZee.append([Tb4[0, 3], Tb4[1, 3], Tb4[2, 3]])
# Conversione in array NumPy per plotting
XYZ1 = np.asarray(XYZ1)
XYZ2 = np.asarray(XYZ2)
XYZ3 = np.asarray(XYZ3)
XYZee = np.asarray(XYZee)

In [ ]:
import matplotlib.pyplot as plt

# Inizializzazione dei log
G_log = []
tau_v_log = []

# Calcolo delle grandezze dinamiche lungo la traiettoria
for q, qd in zip(Q_, Qdot_):
    G_val = G_expr_np(q[0], q[1], q[2], q[3])[0]
    tau_v_val = tau_v_expr_np(qd[0], qd[1], qd[2], qd[3])
    G_log.append(G_val)
    tau_v_log.append(tau_v_val)

# Conversione in array NumPy
G_log = np.array(G_log)
tau_v_log = np.array(tau_v_log)

# -------------------------
# Creazione del subplot
# -------------------------
fig, axs = plt.subplots(2, 2, figsize=(10, 5))
fig.suptitle("Analisi della Dinamica del Manipolatore", fontsize=16)

# -------------------------
# Colonna Sinistra
# -------------------------

# Traiettoria End-effector: x, y, z nel tempo
axs[0, 0].plot(time, XYZee[:, 0], label=r"$x(t)$")
axs[0, 0].plot(time, XYZee[:, 1], label=r"$y(t)$")
axs[0, 0].plot(time, XYZee[:, 2], label=r"$z(t)$")
axs[0, 0].set_title(r"Traiettoria End-Effector nello Spazio")
axs[0, 0].set_xlabel("Tempo [s]")
axs[0, 0].set_ylabel("Posizione [m]")
axs[0, 0].grid()
axs[0, 0].legend(loc='center left', bbox_to_anchor=(1.05, 0.5))

# Spazio dei giunti: q1–q4(t)
axs[1, 0].plot(time, np.array(Q_)[:, 0], label=r"$q_1(t)$")
axs[1, 0].plot(time, np.array(Q_)[:, 1], label=r"$q_2(t)$")
axs[1, 0].plot(time, np.array(Q_)[:, 2], label=r"$q_3(t)$")
axs[1, 0].plot(time, np.array(Q_)[:, 3], label=r"$q_4(t)$")
axs[1, 0].set_title("Traiettoria nello Spazio dei Giunti")
axs[1, 0].set_xlabel("Tempo [s]")
axs[1, 0].set_ylabel("Posizione [rad]")
axs[1, 0].grid()
axs[1, 0].legend(loc='center left', bbox_to_anchor=(1.05, 0.5))

# -------------------------
# Colonna Destra:
# -------------------------

# Coppie gravitazionali G1–G4
axs[0, 1].plot(time, G_log[:, 0], label=r"$G_1$")
axs[0, 1].plot(time, G_log[:, 1], label=r"$G_2$")
axs[0, 1].plot(time, G_log[:, 2], label=r"$G_3$")
axs[0, 1].plot(time, G_log[:, 3], label=r"$G_4$")
axs[0, 1].set_title("Coppie Gravitazionali $G(q)$")
axs[0, 1].set_xlabel("Tempo [s]")
axs[0, 1].set_ylabel("Coppia [Nm]")
axs[0, 1].grid()
axs[0, 1].legend(loc='center left', bbox_to_anchor=(1.05, 0.5))

# Coppie di attrito viscoso
axs[1, 1].plot(time, tau_v_log[:, 0], label=r"$\tau_{v1}$")
axs[1, 1].plot(time, tau_v_log[:, 1], label=r"$\tau_{v2}$")
axs[1, 1].plot(time, tau_v_log[:, 2], label=r"$\tau_{v1}$")
axs[1, 1].plot(time, tau_v_log[:, 3], label=r"$\tau_{v2}$")
axs[1, 1].set_title("Coppie di Attrito Viscoso $\tau_v$")
axs[1, 1].set_xlabel("Tempo [s]")
axs[1, 1].set_ylabel("Coppia [Nm]")
axs[1, 1].grid()
axs[1, 1].legend(loc='center left', bbox_to_anchor=(1.05, 0.5))

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D

plt.ioff()  # disattiva la modalità interattiva

# Creo la figura e l'asse 3D
fig = plt.figure(figsize=(6, 6))
ax  = fig.add_subplot(111, projection='3d')

def animate(i):
    ax.clear()

    # --------------------------------------------------
    # Disegno i link e i giunti in 3D
    # --------------------------------------------------

    # Link 1: da (0,0,0) a XYZ1[i]
    ax.plot(
        [0, XYZ1[i, 0]],
        [0, XYZ1[i, 1]],
        [0, XYZ1[i, 2]],
        lw=5, color='red', label='Link 1'
    )
    ax.scatter([0], [0], [0], marker='o', s=30, color='black')

    # Link 2: da XYZ1[i] a XYZ2[i]
    ax.plot(
        [XYZ1[i, 0],   XYZ2[i, 0]],
        [XYZ1[i, 1],   XYZ2[i, 1]],
        [XYZ1[i, 2],   XYZ2[i, 2]],
        lw=5, color='blue', label='Link 2'
    )
    ax.scatter(XYZ1[i, 0], XYZ1[i, 1], XYZ1[i, 2], marker='o', s=30, color='black')

    # Link 3: da XYZ2[i] a XYZ3[i]
    ax.plot(
        [XYZ2[i, 0],   XYZ3[i, 0]],
        [XYZ2[i, 1],   XYZ3[i, 1]],
        [XYZ2[i, 2],   XYZ3[i, 2]],
        lw=5, color='orange', label='Link 3'
    )
    ax.scatter(XYZ2[i, 0], XYZ2[i, 1], XYZ2[i, 2], marker='o', s=30, color='black')

    # Link 4: da XYZ3[i] a XYZee[i]
    ax.plot(
        [XYZ3[i, 0],   XYZee[i, 0]],
        [XYZ3[i, 1],   XYZee[i, 1]],
        [XYZ3[i, 2],   XYZee[i, 2]],
        lw=5, color='green', label='Link 4'
    )
    ax.scatter(XYZ3[i, 0], XYZ3[i, 1], XYZ3[i, 2], marker='o', s=30, color='black')

    # --------------------------------------------------
    # Disegno le tracce
    # --------------------------------------------------

    ax.plot(
        XYZ1[:i+1, 0], XYZ1[:i+1, 1], XYZ1[:i+1, 2],
        lw=1, color='red', alpha=0.5, label='Tr. Giunto 1'
    )
    ax.plot(
        XYZ2[:i+1, 0], XYZ2[:i+1, 1], XYZ2[:i+1, 2],
        lw=1, color='blue', alpha=0.5, label='Tr. Giunto 2'
    )
    ax.plot(
        XYZ3[:i+1, 0], XYZ3[:i+1, 1], XYZ3[:i+1, 2],
        lw=1, color='orange', alpha=0.5, label='Tr. Giunto 3'
    )
    ax.plot(
        XYZee[:i+1, 0], XYZee[:i+1, 1], XYZee[:i+1, 2],
        lw=1, color='green', alpha=0.5, label='Tr. End-Effector'
    )

    # --------------------------------------------------
    # Impostazioni grafiche
    # --------------------------------------------------

    ax.set_xlabel(r'$x$ [m]', fontsize=12)
    ax.set_ylabel(r'$y$ [m]', fontsize=12)
    ax.set_zlabel(r'$z$ [m]', fontsize=12)
    ax.set_title("Animazione Manipolatore", fontsize=13)

    lim = l1 + l2 + l3 + l4 + 0.05
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_zlim(-lim, lim)
    ax.set_box_aspect((1, 1, 1))   # scala uguale su x, y e z
    ax.grid(True)

    # Mostro il time‐stamp in alto a sinistra
    ax.text(
        -lim + 0.05, lim - 0.05, lim - 0.05,
        f"t = {time[i]:.2f} s",
        fontsize=10,
        bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3')
    )

    ax.legend(loc='center left', bbox_to_anchor=(1.05, 0.5), fontsize=8)

    plt.tight_layout()
ani = animation.FuncAnimation(fig, animate,
                              frames=range(0, len(time), 5),
                              interval=50)

from IPython.display import HTML
HTML(ani.to_jshtml())

plt.show()